In [18]:
API_KEY = "***"
url = f"http://128.118.54.16:3030/api/openai/{API_KEY}"

In [14]:




import json


def request(url):

    payload = {

        "model": "gpt-5",

        "messages": [

            {"role": "user", "content": "write a story about geography"}

        ],

        "stream": True,

        # "temperature": 0.8

    }

    response = requests.post(url, json=payload, stream=True)



    if response.status_code == 200:

        for line in response.iter_lines():

            if line:

                line = line.decode('utf-8')

                if line.startswith('data: '):

                    data_str = line[6:]



                    if data_str == '[DONE]':

                        break

                    try:

                        chunk = json.loads(data_str)

                        if 'choices' in chunk and len(chunk['choices']) > 0:

                            delta = chunk['choices'][0].get('delta', {})

                            content = delta.get('content', '')

                            if content:

                                print(content, end='', flush=True)

                    except json.JSONDecodeError:

                        pass

    else:

        print(f"Error: {response.text}")


request(url)


Assistant Response:
I was born before I had a name, stitched from snowmelt and the breath of lichens, from the quiet unlacing of winter on a mountain that had lifted its forehead into the sky long before anyone measured heights. The mountain was not always a mountain. It came from patient collisions far beneath it—plates shouldering plates, rock wrinkling like a rug being pushed from below. I remember none of that in the way you remember childhood, but I carry the proof: ground ground to powder by the old glaciers, silt the color of smoke in my first bright water, the stones under me small and round as coins from centuries of use.

I slipped downward at first, shy and narrow, glancing off granite, trembling when sun loosened the last ropes of ice and sent me skittering in a hundred threads. Moss cheered me in schoolyard greens. Marmots craned their whiskers to listen. I learned the language of slope and fall, and how gravity prods the smallest water with a constant finger, always towa

In [ ]:
def Query_tuning(user_query, model_name, stream):
    """
    Fine-tune user query using the specified model.
    Supports both OpenAI and local models via unified provider.
    """
    try:
        # Check if this is a local model
        import SpatialAnalysisAgent_ModelProvider as ModelProvider
        provider_name = ModelProvider.ModelProviderFactory._model_providers.get(model_name, 'openai')

        if provider_name == 'ollama':
            # Use local model with LangChain ChatOpenAI pointing to local server
            llm = ChatOpenAI(
                base_url="http://128.118.54.16:11434/v1",
                api_key="no-api",
                model_name=model_name,
                openai_api_key="no-api"
            )
        else:
            # Use OpenAI model
            OpenAI_key = load_OpenAI_key()
            from SpatialAnalysisAgent_ModelProvider import create_unified_client
            client, provider = create_unified_client(model_name)
            # Prepare messages (OpenAI-style)
            messages = [
                {"role": "system", "content": constants.cot_description_prompt},
                {"role": "user", "content": user_query}
            ]
            response = provider.generate_completion(
                client,
                model_name,
                messages,
                stream=stream
                )
            # Check if it's a generator (streaming response)
            if hasattr(response, '__iter__') and not hasattr(response, 'choices'):
                # It's a streaming generator, collect all tokens AND print them
                fine_tuned_request = ""
                for chunk in response:
                    if chunk:
                        print(chunk, end="")  # Print each chunk as it arrives
                        fine_tuned_request += chunk
                print()  # New line after streaming is complete
            else:
                # It's a regular response object
                if hasattr(response, "choices"):
                    fine_tuned_request = response.choices[0].message.content.strip()
                elif isinstance(response, dict):
                    fine_tuned_request = response.get("choices", [{}])[0].get("message", {}).get("content", "")
                else:
                    fine_tuned_request = str(response)

    except ImportError:

        # Fallback to direct OpenAI SDK usage

        OpenAI_key = load_OpenAI_key()
        client = create_openai_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": constants.cot_description_prompt},
                {"role": "user", "content": user_query}
            ]
        )


        fine_tuned_request = response.choices[0].message.content
    return fine_tuned_request


In [ ]:
def Query_tuning(user_query, model_name, stream):
    """Return a fine-tuned prompt using the selected model (streaming or not)."""
    messages = [
        {"role": "system", "content": constants.cot_description_prompt},
        {"role": "user", "content": user_query},
    ]
    try:
        # Unified path (handles OpenAI, local, proxies via your provider)
        from SpatialAnalysisAgent_ModelProvider import create_unified_client
        client, provider = create_unified_client(model_name)
        response = provider.generate_completion(client, model_name, messages, stream=stream)
        return streaming_openai_response(response)
    except ImportError:
        # Direct OpenAI fallback
        from openai import OpenAI
        client = OpenAI(api_key=load_OpenAI_key())
        response = client.chat.completions.create(model=model_name, messages=messages, stream=stream)
        return streaming_openai_response(response)

In [ ]:
# Add this function to generate the task name using UNIFIED MODEL PROVIDER
def generate_task_name_with_model_provider(model_name, task_description):
    prompt = f"Given the following task description: '{task_description}',give the best task that represents this task.\n\n" + \
             f"Provide the task name in just one or two words. \n\n" + \
             f"Underscore '_' is the only alphanumeric symbols that is allowed in a task name. A task_name must not contain quotations or inverted commas example or space. \n"

    # Use the unified model provider
    try:
        from SpatialAnalysisAgent_ModelProvider import create_unified_client
        client, provider = create_unified_client(model_name)

        # Generate response using the provider
        response = provider.generate_completion(
            client,
            model_name,
            [{"role": "user", "content": prompt}],
            stream=False
        )
    except ImportError:
        # Fallback to basic OpenAI client
        client = create_openai_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "user", "content": prompt},
            ]
        )

    task_name = response.choices[0].message.content
    return task_name